## Centerline

In [25]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk
from sklearn.decomposition import PCA
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import BSpline
from scipy import ndimage
import trimesh
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, getSurfaceMesh, tetra_mesh_from_stl, set_the_offset

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/"

eyeball_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/Segmentation_1_surf_id_1_inner.stl"
lense_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/re-meshed_h_1.0/id_7__h_1.0.stl"

offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")
pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

out_folder = "extended_muscles_1/"
centerline_folder = "centerlines/"

mask_filename_base = "Segmentation_1_filtered_surf_id_" # + _1.npy

muslce_ids = [2,3,4,5]

eyeball_id = 1
lense_id = 7

mask_eyeball = numpy.load(base + mask_filename_base + str(eyeball_id) + ".npy")



## 1. Lense centre & axis

In [26]:
mesh = trimesh.load(lense_mesh)
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


projections_0 = (mesh.vertices - lense_center) @ vh[0]
lense_diameter = (projections_0.max() - projections_0.min())*pixel_spacing[0]

projections_1 = (mesh.vertices - lense_center) @ vh[-1]
lense_thickness = (projections_1.max() - projections_1.min())*pixel_spacing[0]



print("lense_diameter = ", lense_diameter)


def eyeball_axis(t):
    return lense_center + t * axis_direction


axis_points = [eyeball_axis(-20.0), eyeball_axis(20.0)]

export_centerline(axis_points, 1.0 , base + out_folder + "axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

lense_center =  [ 1.01680029  8.08340236 -2.5252567 ]
lense_diameter =  3.9438819228328748


## 2. Fit two planes

In [27]:
def load_centerline(base, centerline, cid):
    data = numpy.load(base + centerline + "centerline_id_"+str(cid)+"_parametric.npz")

    
    sx = BSpline(data["tx"], data["cx"], data["kx"])
    sy = BSpline(data["ty"], data["cy"], data["ky"])
    sz = BSpline(data["tz"], data["cz"], data["kz"])
    
    return sx, sy, sz

centerlines = [load_centerline(base, centerline_folder, cid) for cid in muslce_ids]

In [28]:
def sample_spline(sx, sy, sz, n=100):
    t_min = max(sx.t[0], sy.t[0], sz.t[0])
    t_max = min(sx.t[-1], sy.t[-1], sz.t[-1])
    
    t = numpy.linspace(t_min, t_max, n)
    
    x = sx(t)
    y = sy(t)
    z = sz(t)
    
    return numpy.vstack((x, y, z)).T  # shape (n, 3)

def fit_plane(points):
    # points: (N, 3)
    centroid = points.mean(axis=0)
    
    # subtract centroid
    X = points - centroid
    
    # SVD
    _, _, vh = numpy.linalg.svd(X)
    
    normal = vh[-1]  # smallest singular vector
    
    return centroid, normal


pairs = [(0, 2), (1, 3)]

planes = []

for i, j in pairs:
    pts1 = sample_spline(*centerlines[i])
    pts2 = sample_spline(*centerlines[j])
    
    all_pts = numpy.vstack((pts1, pts2, lense_center[None, :]))
    
    point, normal = fit_plane(all_pts)
    
    planes.append((point, normal))


print(numpy.dot(planes[0][1], planes[1][1] ))

0.06065041362809068


In [29]:
n3 = numpy.cross(planes[0][1], planes[1][1])
n3 = n3 / numpy.linalg.norm(n3)



## 3. Eyeball section 

In [30]:
mesh_eye = trimesh.load(eyeball_mesh)

mesh_eye.vertices = mesh_eye.vertices - offset

side = (mesh_eye.vertices - lense_center) @ axis_direction
eps = 1e-6
labels = numpy.zeros(len(mesh_eye.vertices))
labels[side > eps] = 1
labels[side < -eps] = 0

print(labels.shape)

meshio.write(
    base + out_folder + "eyeball_with_labels.vtk",
    meshio.Mesh(
        points=mesh_eye.vertices,
        cells=[("triangle", mesh_eye.faces)],
        point_data={"side": labels}
    )
)

(12080,)


In [31]:
section = mesh_eye.section(
    plane_origin=lense_center - lense_thickness*axis_direction,
    plane_normal=axis_direction
)


section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + out_folder + "section_curve.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)


## Map the cross sections

In [32]:
# --------------------------------------------------
# 1. Build local frame from tangent
# --------------------------------------------------
def build_frame(tangent):
    tangent = tangent / numpy.linalg.norm(tangent)
    
    tmp = numpy.array([1.0, 0.0, 0.0])
    if abs(numpy.dot(tmp, tangent)) > 0.9:
        tmp = numpy.array([0.0, 1.0, 0.0])
    
    u = numpy.cross(tangent, tmp)
    u /= numpy.linalg.norm(u)
    
    v = numpy.cross(tangent, u)
    
    return u, v, tangent

# --------------------------------------------------

def fit_ellipse(section):
    X = section - section.mean(axis=0)
    
    _, S, vh = numpy.linalg.svd(X)
    
    axes = vh          # directions
    scales = S / numpy.sqrt(len(section))  # approximate radii
    
    return axes, scales

# --------------------------------------------------

muscle_i = 3

mask_muscle = numpy.load(base + mask_filename_base + str(muslce_ids[muscle_i]) + ".npy")
pts = sample_spline(*centerlines[muscle_i])


n_samples = 10
indices = numpy.linspace(5, len(pts)-5, n_samples).astype(int)


cross_sections = []
centers = []

for idx in indices:
    
    p = pts[idx]
    p_vox = numpy.round(p).astype(int)
    
    # tangent
    tangent = pts[idx+1] - pts[idx-1]
    tangent /= numpy.linalg.norm(tangent)
    
    u, v, _ = build_frame(tangent)
    
    section_pts = []
    
    radius = 50
    
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            for dz in range(-radius, radius+1):
                
                q = p_vox + numpy.array([dx, dy, dz])
                
                if (
                    0 <= q[0] < mask_muscle.shape[0] and
                    0 <= q[1] < mask_muscle.shape[1] and
                    0 <= q[2] < mask_muscle.shape[2]
                ):
                    if mask_muscle[tuple(q)] == 1:
                        
                        rel = q - p_vox
                        
                        x = numpy.dot(rel, u)
                        y = numpy.dot(rel, v)
                        
                        section_pts.append([x, y])
    
    if len(section_pts) > 10:
        cross_sections.append(numpy.array(section_pts))
        centers.append(idx)

# elipse
ellipse_params = [fit_ellipse(cs) for cs in cross_sections]


# evolutiom
r1 = numpy.array([p[1][0] for p in ellipse_params])
r2 = numpy.array([p[1][1] for p in ellipse_params])
t_vals = numpy.linspace(1, 1.9, len(r1))

coef_r1 = numpy.polyfit(t_vals, r1, deg=2)
coef_r2 = numpy.polyfit(t_vals, r2, deg=2)


## Muscle extention

In [33]:
def closest_point_on_curve(p_end, curve):
    diff = curve - p_end
    dist2 = numpy.sum(diff**2, axis=1)
    idx = numpy.argmin(dist2)
    return curve[idx], idx

def find_intersection_with_mask(start, direction, mask, step=0.5, max_dist=100):
    p = start.copy()
    
    for _ in range(int(max_dist / step)):
        p += direction * step
        idx = numpy.round(p).astype(int)
        
        # check bounds
        if (
            0 <= idx[0] < mask.shape[0] and
            0 <= idx[1] < mask.shape[1] and
            0 <= idx[2] < mask.shape[2]
        ):
            if mask[tuple(idx)] == 1:
                return p  # first contact
        else:
            break
    
    return None


p_end = pts[-1]*pixel_spacing - offset # last point of spline (3D)

print("p_end = ", p_end)

target, idx = closest_point_on_curve(p_end, section.vertices)

print("target = ", target)





p_end =  [-14.32199832   1.61848542  -0.7513497 ]
target =  [-11.43423471   6.65670148  -2.08016658]


In [34]:
def bezier_quadratic(P0, P1, P2, n=50):
    t = numpy.linspace(0, 1, n)
    return (
        ((1 - t)**2)[:, None] * P0 +
        (2 * (1 - t) * t)[:, None] * P1 +
        (t**2)[:, None] * P2
    )

tangent = pts[-1] - pts[-2]
tangent = tangent / numpy.linalg.norm(tangent)

alpha = 8  # controls curvature strength

P0 = p_end
P2 = target
P1 = P0 + alpha * tangent

extension = bezier_quadratic(P0, P1, P2)

## Option 1: Extended muscle shape

In [35]:
# --------------------------------------------------
# CORRECTED EVOLVING EXTENSION
# --------------------------------------------------

# 1. FIX AXIS SWAPPING: Identify which radius goes with which vector
# In fit_ellipse, scales[0] corresponds to vh[0] (major) and scales[1] to vh[1] (minor)
last_axes, last_scales = ellipse_params[-1]

# We ensure r1_ext always scales the first axis of last_axes, and r2_ext the second.
# This keeps the orientation locked regardless of which radius is technically 'larger'.
v1 = last_axes[0] 
v2 = last_axes[1]

# 2. FIX CONTINUITY: Ensure t_ext starts exactly after the original t_vals
# Original was linspace(0, 1, num_samples). Extension must start at 1.0.
# If your extension has 50 points and represents a 30% length increase:
t_ext = numpy.linspace(1.0, 1.3, len(extension))

r1_ext = numpy.polyval(coef_r1, t_ext)
r2_ext = numpy.polyval(coef_r2, t_ext)

# Safety to prevent the muscle from "vanishing" or inverted geometry
r1_ext = numpy.maximum(r1_ext, 0.25)
r2_ext = numpy.maximum(r2_ext, 0.25)

new_mask = numpy.zeros_like(mask_eyeball)

for i, p in enumerate(extension):
    p_vox = (p + offset) / pixel_spacing
    
    # Calculate Tangent
    if len(extension) > 1:
        if i == 0: tangent = extension[i+1] - extension[i]
        elif i == len(extension)-1: tangent = extension[i] - extension[i-1]
        else: tangent = extension[i+1] - extension[i-1]
    else:
        tangent = numpy.array([0, 0, 1])
        
    u, v, _ = build_frame(tangent)
    
    # Use evolving radii for this specific step
    a = r1_ext[i]
    b = r2_ext[i]
    
    n_angles = 50
    n_radials = 10 
    
    for r_factor in numpy.linspace(0, 1, n_radials):
        for theta in numpy.linspace(0, 2 * numpy.pi, n_angles):
            # 3. MANUAL RECONSTRUCTION (Avoids the 'swapped axis' SVD flip)
            # We use the raw trig values to scale our locked vectors v1 and v2
            # then project those into the local 3D u,v frame.
            
            # Unit coordinates in the ellipse's own 2D space
            x_el = r_factor * numpy.cos(theta)
            y_el = r_factor * numpy.sin(theta)
            
            # Scale by the trend-calculated radii a and b
            # v1 and v2 are the 2D directions captured from the last real slice
            p_local_2d = x_el * a * v1 + y_el * b * v2
            
            # Project to 3D
            point = p_vox + p_local_2d[0] * u + p_local_2d[1] * v
            idx = numpy.round(point).astype(int)
            
            if (0 <= idx[0] < new_mask.shape[0] and
                0 <= idx[1] < new_mask.shape[1] and
                0 <= idx[2] < new_mask.shape[2]):
                new_mask[tuple(idx)] = 1

# Save the evolving mask
output_path = base + out_folder + "mask_extention_" + str(muslce_ids[muscle_i]) + ".npy"
numpy.save(output_path, new_mask)

## Option 0 : circular cross-section

In [36]:
h=1

new_mask = ndimage.median_filter(new_mask, size=3) 


mesh_surf = getSurfaceMesh(new_mask, base + out_folder +"_extention_surf_id_"+ str(muslce_ids[muscle_i]) + ".stl", pixel_spacing[0], False )

tetra_mesh_from_stl(base + out_folder + "_extention_surf_id_"+ str(muslce_ids[muscle_i]) +".stl", base + out_folder + "_extention_surf_id_" + str(muslce_ids[muscle_i]) + "_h_"+str(h), h)
set_the_offset(base  + out_folder + "_extention_surf_id_" + str(muslce_ids[muscle_i]) + "_h_"+str(h), offset)

Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles_1/_extention_surf_id_4.stl
Info    : Reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles_1/_extention_surf_id_4.stl'...
Info    : 1388 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles_1/_extention_surf_id_4.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 1388 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.0122984s, CPU 0.012231s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry o

Warning: STL can only write triangle cells. Discarding vertex, tetra, line.